# VAE & Latent Space

In this notebook we explore the **Video Variational Autoencoder (VAE)** -- the component that compresses raw pixel-space video into a compact latent representation suitable for diffusion.

## Why do we need a VAE?

Raw video data is enormous. A single 16-frame clip at 64x64 resolution already contains **196,608** floating-point values (3 channels x 16 frames x 64 x 64). Running diffusion at full resolution would be extremely slow and memory-hungry.

The VAE solves this by learning a compressed representation:

- **Encoder**: maps `[B, 3, T, H, W]` to a low-dimensional latent `[B, C_lat, T', H', W']`
- **Decoder**: reconstructs the video from the latent back to `[B, 3, T, H, W]`

The diffusion model operates entirely in latent space, and only the decoder is needed at the very end to produce the final video.

### The real Wan VAE

Wan 2.1's VAE is a sophisticated 3D convolutional network:

- Uses **CausalConv3d** layers that respect temporal causality (frame $t$ cannot see frame $t+1$)
- 4 downsampling stages with **ResidualBlocks** and **RMSNorm**
- **8x spatial downsampling** (64 -> 8) and **4x temporal downsampling** (16 -> 4)
- **16 latent channels** with mean/std normalization
- Middle block with self-attention for global context
- Reparameterization trick: $z = \mu + \sigma \cdot \epsilon$ where $\epsilon \sim \mathcal{N}(0, I)$

### Our DummyVAE

Our simplified DummyVAE preserves the **same dimensional concept** with much simpler internals:

- Simple `Conv3d` encoder/decoder (no causal convolutions)
- **4x spatial** and **4x temporal** downsampling
- **4 latent channels** (fewer than Wan's 16, to keep the nano model small)
- Same reparameterization trick
- Same latent normalization (mean/std scaling)

In [ ]:
import sys
sys.path.insert(0, '/fs04/scratch2/kl27/Ash/projs/Claude_projects/tutorial_GenAI/video_gen/nano-video-gen')

import torch
import matplotlib.pyplot as plt

from nano_video_gen.model.nano_vae import DummyVAE
from nano_video_gen.visualization import visualize_latent_space

# Instantiate the DummyVAE
vae = DummyVAE(
    in_channels=3,
    latent_channels=4,
    spatial_factor=4,
    temporal_factor=4,
)

# Print the architecture
print("=== DummyVAE Architecture ===")
print(vae)
print()

# Count parameters
total_params = sum(p.numel() for p in vae.parameters())
trainable_params = sum(p.numel() for p in vae.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# Create a random "video" tensor and run encode -> decode
torch.manual_seed(42)

# Simulate a video: [B, C, T, H, W] with values in [-1, 1]
video = torch.randn(1, 3, 16, 64, 64).clamp(-1, 1)
print(f"Input video shape:          {list(video.shape)}")
print(f"Input values per sample:    {video[0].numel():,}")
print()

# Encode to latent space
vae.eval()
with torch.no_grad():
    latent = vae.encode(video)

print(f"Latent shape:               {list(latent.shape)}")
print(f"Latent values per sample:   {latent[0].numel():,}")
print(f"Compression ratio:          {video[0].numel() / latent[0].numel():.1f}x")
print()

# Decode back to pixel space
with torch.no_grad():
    reconstruction = vae.decode(latent)

print(f"Reconstruction shape:       {list(reconstruction.shape)}")
print(f"Shapes match:               {video.shape == reconstruction.shape}")

In [ ]:
# Visualize the latent channels
# Each channel captures different aspects of the video content

fig = visualize_latent_space(
    latent,
    title="DummyVAE Latent Channels (first temporal frame)"
)
plt.show()

## Reconstruction Quality

Our DummyVAE is intentionally simple -- it uses only a few convolutional layers without residual blocks, attention, or causal convolutions. As a result, its reconstruction quality will be poor compared to the real Wan VAE.

However, the **concept** is exactly the same:

1. **Encode**: compress the video into a low-dimensional latent
2. **Latent space**: the diffusion model operates here
3. **Decode**: reconstruct the full-resolution video from the denoised latent

In a real system, the VAE is trained separately (with reconstruction loss + KL divergence) until it can faithfully compress and reconstruct videos. The diffusion model is then trained in the frozen VAE's latent space.

Let's measure the reconstruction error to see how much information our simple VAE loses:

In [ ]:
# Compute reconstruction error
with torch.no_grad():
    mse = torch.nn.functional.mse_loss(reconstruction, video)
    mae = torch.nn.functional.l1_loss(reconstruction, video)
    # PSNR (Peak Signal-to-Noise Ratio)
    # For data in [-1, 1], max value range is 2
    psnr = 10 * torch.log10(4.0 / mse)  # 4 = (2)^2, since range is [-1,1]

print("=== Reconstruction Error (untrained DummyVAE) ===")
print(f"MSE  (Mean Squared Error):    {mse.item():.4f}")
print(f"MAE  (Mean Absolute Error):   {mae.item():.4f}")
print(f"PSNR (Peak SNR, dB):          {psnr.item():.2f} dB")
print()
print("Note: An untrained VAE will have high error. After training,")
print("a good VAE achieves PSNR > 30 dB (nearly imperceptible loss).")
print()

# Visualize original vs reconstruction (first frame)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

orig_frame = video[0, :, 0].permute(1, 2, 0).clamp(-1, 1)
orig_frame = (orig_frame + 1) / 2  # to [0, 1]

recon_frame = reconstruction[0, :, 0].permute(1, 2, 0).clamp(-1, 1)
recon_frame = (recon_frame + 1) / 2

axes[0].imshow(orig_frame.numpy())
axes[0].set_title('Original (frame 0)')
axes[0].axis('off')

axes[1].imshow(recon_frame.numpy())
axes[1].set_title(f'Reconstruction (PSNR={psnr.item():.1f} dB)')
axes[1].axis('off')

plt.suptitle('VAE Encode -> Decode', fontsize=14)
plt.tight_layout()
plt.show()

## DummyVAE vs Real Wan VAE Architecture

The table below compares our simplified DummyVAE with the real Wan VAE architecture:

| Feature | Real Wan VAE | DummyVAE |
|---------|-------------|----------|
| Convolution type | CausalConv3d (temporal causality) | Standard Conv3d |
| Encoder stages | 4 stages with ResBlocks | 3 Conv layers |
| Normalization | RMSNorm in ResBlocks | None (SiLU only) |
| Middle block | ResBlock + AttentionBlock + ResBlock | None |
| Spatial downsample | 8x (64 -> 8) | 4x (64 -> 16) |
| Temporal downsample | 4x (16 -> 4) | 4x (16 -> 4) |
| Latent channels | 16 | 4 |
| Channel progression | 96 -> 192 -> 384 -> 384 | 3 -> 32 -> 64 -> 8 |
| Reparameterization | Yes ($z = \mu + \sigma \cdot \epsilon$) | Yes (same) |
| Latent normalization | Mean/std scaling | Mean/std scaling |
| Decoder | Symmetric with upsampling | Symmetric with ConvTranspose3d |
| Output activation | None | Tanh (clamp to [-1, 1]) |
| Parameters | ~100M+ | ~25K |

The key takeaway: our DummyVAE captures the **same encode-latent-decode paradigm** that makes latent diffusion work, even though it lacks the capacity for high-quality reconstruction. In a production system, the VAE is a critical component trained to very high fidelity.